In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys
from openpyxl import Workbook
import time

In [2]:
#Entry
tperiod = 12

# Distribute evacuees to shelters

In [4]:
def allocate_demand(distance_file, demand_files, output_file, num_scenarios=100):
    
    # Load distance matrix (7 PODs, 21 Shelters, No Headers)
    distance_matrix = pd.read_excel(distance_file, header=None).iloc[:7, :21]

    # Compute inverse distance weights (avoid division by zero)
    inverse_distances = 1 / distance_matrix.replace(0, float("inf"))
    weights = inverse_distances.div(inverse_distances.sum(axis=1), axis=0)  # Normalize weights

    # Load demand data (7 files, each with 1048 scenarios & 6 time periods)
    demand_scenarios = [pd.read_csv(file, header=None) for file in demand_files]

    # Allocate demand for the specified number of scenarios
    allocated_demand = {}
    for scenario_idx in range(num_scenarios):
        # Extract demand for this scenario across all 7 PODs (6 time periods each)
        demand_df = pd.DataFrame({t: [demand_scenarios[pod].iloc[scenario_idx, t] for pod in range(7)] for t in range(6)})

        # Allocate demand based on distance weights
        allocation_matrix = {t: weights.multiply(demand_df[t], axis=0).sum(axis=0) for t in range(6)}

        # Convert to DataFrame where rows are shelters, columns are time periods
        allocated_demand[f"Scenario_{scenario_idx}"] = pd.DataFrame(allocation_matrix, index=range(21))

    # Save the results
    with pd.ExcelWriter(output_file) as writer:
        for scenario, df in allocated_demand.items():
            df.to_excel(writer, sheet_name=scenario)

    print(f"✅ Demand allocation completed. File saved: {output_file}")

# Example Usage:
distance_file = "distanceFull-podtoshelter.xlsx"
demand_files = [f"DP_{i}.csv" for i in range(7)]
output_file = "Shelter_Demand.xlsx"

allocate_demand(distance_file, demand_files, output_file, num_scenarios=100)

✅ Demand allocation completed. File saved: Shelter_Demand.xlsx


# Calculate PoD demand

In [6]:
def calculate_landfall_demand_fixed(wind_speed_file, pod_info_file, output_file):
    # Load the data
    wind_speed_df = pd.read_excel(wind_speed_file, header=None)
    pod_info_df = pd.read_excel(pod_info_file, sheet_name='Sheet1')

    # Read hurricane categories from column I (index 8)
    categories = wind_speed_df.iloc[:100, 8]

    # Define columns corresponding to each category
    category_columns = {
        "Category 1": "I*p_c (cat 1)",
        "Category 2": "I*p_c (cat 2)",
        "Category 3": "I*p_c (cat 3)",
        "Category 4": "I*p_c (cat 4)",
        "Category 5": "I*p_c (cat 5)"
    }

    # Get the column for '% 1- (Likely to evacuate)'
    evac_column = '% 1- (Likely to evacuate)'

    # Create an Excel writer to save the results
    with pd.ExcelWriter(output_file) as writer:
        for scenario_idx in range(100):
            # Get the category for this scenario
            category = categories.iloc[scenario_idx]

            # Check if the category is valid
            demand_column = category_columns.get(category, None)

            if demand_column is not None:
                # Extract the necessary data for calculation
                pod_demand_values = pod_info_df[demand_column]
                evacuation_factors = pod_info_df[evac_column]

                # Calculate demand: (I*p_c) * (% 1- Likely to evacuate)
                demand_values = pod_demand_values * evacuation_factors

            else:
                # If the category is N/A, set demand to 0
                demand_values = pd.Series(0, index=pod_info_df.index)

            # Ensure numeric data type
            demand_values = demand_values.astype(float)

            # Prepare the output DataFrame
            scenario_data = pd.DataFrame({
                'County': pod_info_df['County'],
                'total demand': demand_values
            })

            # Write to Excel sheet for this scenario
            sheet_name = f'Scenario_{scenario_idx}'
            scenario_data.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"✅ Corrected output file generated: {output_file}")

# Example usage of the function
wind_speed_file = 'wind_speed_err.xlsx'
pod_info_file = 'PoD location information.xlsx'
output_file = 'PoD_Demand_total.xlsx'

calculate_landfall_demand_fixed(wind_speed_file, pod_info_file, output_file)


✅ Corrected output file generated: PoD_Demand_total.xlsx


In [7]:
def distribute_demand_by_time(total_demand_file, output_file):
    # Load the total demand file
    total_demand_data = pd.ExcelFile(total_demand_file)

    # Define time periods and normal distribution parameters
    time_periods = [6, 7, 8, 9, 10, 11]
    mean = 8  # Peak at time period 8
    std_dev = 1  # Spread of the distribution

    # Calculate normalized weights using a normal distribution
    distribution_values = [np.exp(-((t - mean) ** 2) / (2 * (std_dev ** 2))) for t in time_periods]
    weights = np.array(distribution_values) / sum(distribution_values)  # Normalize weights to sum to 1

    # Create an Excel writer to save the results
    with pd.ExcelWriter(output_file) as writer:
        for sheet_name in total_demand_data.sheet_names:
            # Read the data for the current scenario
            scenario_data = pd.read_excel(total_demand_data, sheet_name=sheet_name)

            # Apply the distribution weights to the total demand for each county
            distributed_demands = {
                f'Time_{t}': scenario_data['total demand'] * weight for t, weight in zip(time_periods, weights)
            }

            # Prepare the output DataFrame
            output_data = scenario_data[['County']].copy()
            for period, demand in distributed_demands.items():
                output_data[period] = demand

            # Write the output data to the Excel sheet
            output_data.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"✅ Distributed demand file generated: {output_file}")

# Example usage of the function
total_demand_file = 'PoD_Demand_total.xlsx'
output_file = 'POD_Demand.xlsx'

distribute_demand_by_time(total_demand_file, output_file)


✅ Distributed demand file generated: POD_Demand.xlsx


# Integrate all demands

In [9]:
def combine_demand_data(shelter_file, pod_file, output_file, num_scenarios=100):
    # Read data from both files (assuming data is in each scenario sheet)
    shelter_data = pd.read_excel(shelter_file, sheet_name=None)
    pod_data = pd.read_excel(pod_file, sheet_name=None)

    # Create an output Excel writer using openpyxl
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for scenario_num in range(num_scenarios):
            scenario_sheet = f'Scenario_{scenario_num}'

            # Get data for the current scenario
            shelter_scenario = shelter_data.get(scenario_sheet, pd.DataFrame())
            pod_scenario = pod_data.get(scenario_sheet, pd.DataFrame())

            # Initialize combined data structure
            combined_data = pd.DataFrame(0.0, index=range(43), columns=range(12))  # Initialize with float values

            # Assign data to the correct time periods and nodes
            if not shelter_scenario.empty:
                combined_data.iloc[6:27, 0:6] = shelter_scenario.iloc[:, 1:7].values  # 21 shelters, periods 0-5

            if not pod_scenario.empty:
                combined_data.iloc[36:43, 6:12] = pod_scenario.iloc[:, 1:7].values  # 7 PODs, periods 6-11

            # Add index column
            combined_data.insert(0, 'Index', range(1, 44))

            # Write combined data to a new sheet
            combined_data.to_excel(writer, sheet_name=scenario_sheet, index=False)

    print(f"✅Combined data saved to {output_file}")

# Example usage:
combine_demand_data('Shelter_Demand.xlsx', 'POD_Demand.xlsx', 'demand_data_original.xlsx')

✅Combined data saved to demand_data_original.xlsx


# Specify the instance version here

In [11]:
# Specify the instance version
instance_version = "original" #Versions: "original", "smallsize", "mediumsize", "largesize"

# Define file names dynamically based on instance version
location_file = f"fullDist Set V Data Location-{instance_version}.xlsx"
county_file = f"fullDist Set C Data-{instance_version}.xlsx"
resource_file = f"fullDist R-{instance_version}.xlsx"
time_matrix_file = f"durationFull-{instance_version}.txt"
distance_matrix_file = f"distanceFull-{instance_version}.txt"
inputs_file = f"fullDistInputs-{instance_version}.xlsx"
k_ijt_file = f"K_ijt_data_fixed-{instance_version}.xlsx"

# Specify the number of scenarios here

In [13]:

# Define the number of scenarios
num_sce = 2 # Example: 10 scenarios (adjust as needed)
sce = list(range(num_sce)) 

# Generate equal probabilities for each scenario
P = [1/num_sce] * num_sce

#Demand data
demand_file = f"demand_data_{instance_version}.xlsx"

# Dynamically generate sheet names based on the number of scenarios
sheet_names = [f"Scenario_{i}" for i in sce]

print(f"Number of scenarios: {num_sce}")
print(f"Generated sheet names: {sheet_names}")
"""

import random

# Define the number of available scenarios in the Excel file (100 in your case)
total_scenarios = 100

# Define the number of scenarios to randomly select
num_sce = 50  # Example: 20 randomly selected scenarios

# Randomly select `num_sce` scenarios from the available 100
random_scenarios = random.sample(range(total_scenarios), num_sce)

# Assign new indices (0 to 19) while keeping track of the actual selected scenarios
sce = list(range(num_sce))  # New indices for the selected scenarios (0 to 19)
selected_sheet_names = [f"Scenario_{i}" for i in random_scenarios]

# Generate equal probabilities for each selected scenario
P = [1 / num_sce] * num_sce

# Demand data file
demand_file = f"demand_data_{instance_version}.xlsx"

# Dynamically generate the new sheet names based on the new indices
sheet_names = [f"Scenario_{i}" for i in sce]

# Print statements
print(f"Number of selected scenarios: {num_sce}")
print(f"Randomly selected real scenario indices: {random_scenarios}")
print(f"Generated sheet names (0 to 19): {sheet_names}")
print(f"Real corresponding sheets: {selected_sheet_names}")
"""

Number of scenarios: 50
Generated sheet names: ['Scenario_0', 'Scenario_1', 'Scenario_2', 'Scenario_3', 'Scenario_4', 'Scenario_5', 'Scenario_6', 'Scenario_7', 'Scenario_8', 'Scenario_9', 'Scenario_10', 'Scenario_11', 'Scenario_12', 'Scenario_13', 'Scenario_14', 'Scenario_15', 'Scenario_16', 'Scenario_17', 'Scenario_18', 'Scenario_19', 'Scenario_20', 'Scenario_21', 'Scenario_22', 'Scenario_23', 'Scenario_24', 'Scenario_25', 'Scenario_26', 'Scenario_27', 'Scenario_28', 'Scenario_29', 'Scenario_30', 'Scenario_31', 'Scenario_32', 'Scenario_33', 'Scenario_34', 'Scenario_35', 'Scenario_36', 'Scenario_37', 'Scenario_38', 'Scenario_39', 'Scenario_40', 'Scenario_41', 'Scenario_42', 'Scenario_43', 'Scenario_44', 'Scenario_45', 'Scenario_46', 'Scenario_47', 'Scenario_48', 'Scenario_49']


'\n\nimport random\n\n# Define the number of available scenarios in the Excel file (100 in your case)\ntotal_scenarios = 100\n\n# Define the number of scenarios to randomly select\nnum_sce = 50  # Example: 20 randomly selected scenarios\n\n# Randomly select `num_sce` scenarios from the available 100\nrandom_scenarios = random.sample(range(total_scenarios), num_sce)\n\n# Assign new indices (0 to 19) while keeping track of the actual selected scenarios\nsce = list(range(num_sce))  # New indices for the selected scenarios (0 to 19)\nselected_sheet_names = [f"Scenario_{i}" for i in random_scenarios]\n\n# Generate equal probabilities for each selected scenario\nP = [1 / num_sce] * num_sce\n\n# Demand data file\ndemand_file = f"demand_data_{instance_version}.xlsx"\n\n# Dynamically generate the new sheet names based on the new indices\nsheet_names = [f"Scenario_{i}" for i in sce]\n\n# Print statements\nprint(f"Number of selected scenarios: {num_sce}")\nprint(f"Randomly selected real scenario 

In [14]:
# Location Data
df_V = pd.read_excel(location_file)
print(df_V.head())

# County Data
df_C = pd.read_excel(county_file)
print(df_C.head())

# Resource Data
df_R = pd.read_excel(resource_file)
print(df_R.head())

# Time and Distance Matrices
timeMatrix = np.loadtxt(time_matrix_file)  # Time duration matrix (seconds)
distanceMatrix = np.loadtxt(distance_matrix_file)  # Distance matrix
print(timeMatrix)
print(distanceMatrix)

# Inputs Data
df_inputs = pd.read_excel(inputs_file)
print(df_inputs.head())

# K_ijt Data
df_K = pd.read_excel(k_ijt_file)
print(df_K.head())

   Index County       Type  Cluster                         Facility  \
0      0    NaN  Warehouse      NaN        SCEMD Winnsboro Warehouse   
1      1    NaN  Warehouse      4.0   US Military Base - Cot Storage   
2      2    NaN   Supplier      NaN                   SOPAKCO (Food)   
3      3    NaN   Supplier      NaN                   Lipsey (Water)   
4      4    NaN   Supplier      NaN  FEMA Warehouse (Food and Water)   

                                             Address   latitude  longitude  \
0              100 Tidewater Dr, Winnsboro, SC 29180 -81.098402  34.403608   
1                Garners Ferry Rd, Hopkins, SC 29061 -80.833969  34.045263   
2                118 S Cypress St, Mullins, SC 29574 -79.262558  34.202395   
3               1701 Oakbrook Dr, Norcross, GA 30093 -84.196174  33.913660   
4  3780 Southside Industrial Pkwy, Atlanta, GA 30354 -84.373590  33.650208   

   Shelter Capacity  Pallet Capacity  ...  P-Kit StartInv  Water StartInv  \
0               NaN  

In [15]:

# Load demand data
d_its = {} # Dictionary to store demand data in (location, time, scenario) format

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0) # Read demand data for the current sheet
    
    # Replace NaN values with 0 (if applicable)
    df = df.fillna(0)
    # Convert DataFrame to a NumPy array for iteration
    demand_array = df.values
    
    # Process demand data for each location and time period
    for location_index in range(demand_array.shape[0]): #Rows represent locations
        for time_period in range(demand_array.shape[1]): #Columns represent time periods
            d_its[(location_index, time_period, s)] = demand_array[location_index, time_period]
"""
# Assume `selected_sheet_names` contains the actual random sheets (e.g., ["Scenario_5", "Scenario_12", ...])
# and `sce` contains the internal scenario indices (0 to 19).

# Load demand data
d_its = {}  # Dictionary to store demand data in (location, time, scenario) format

# Loop through internal indices (0 to 19) but read from the randomly selected sheets
for s, sheet in enumerate(selected_sheet_names):  # `s` is from 0 to 19, but `sheet` is from random scenarios
    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0)  # Read demand data for the current random sheet
    
    # Replace NaN values with 0 (if applicable)
    df = df.fillna(0)
    
    # Convert DataFrame to a NumPy array for iteration
    demand_array = df.values

    # Process demand data for each location and time period
    for location_index in range(demand_array.shape[0]):  # Rows represent locations
        for time_period in range(demand_array.shape[1]):  # Columns represent time periods
            d_its[(location_index, time_period, s)] = demand_array[location_index, time_period]

# Print a sample to verify the correct location
print("Sample demand data for location 5, time 2, scenario 0:", d_its[(7, 2, 0)])
print("Sample demand data for location 6, time 2, scenario 0:", d_its[(1, 0, 1)])
"""

'\n# Assume `selected_sheet_names` contains the actual random sheets (e.g., ["Scenario_5", "Scenario_12", ...])\n# and `sce` contains the internal scenario indices (0 to 19).\n\n# Load demand data\nd_its = {}  # Dictionary to store demand data in (location, time, scenario) format\n\n# Loop through internal indices (0 to 19) but read from the randomly selected sheets\nfor s, sheet in enumerate(selected_sheet_names):  # `s` is from 0 to 19, but `sheet` is from random scenarios\n    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0)  # Read demand data for the current random sheet\n    \n    # Replace NaN values with 0 (if applicable)\n    df = df.fillna(0)\n    \n    # Convert DataFrame to a NumPy array for iteration\n    demand_array = df.values\n\n    # Process demand data for each location and time period\n    for location_index in range(demand_array.shape[0]):  # Rows represent locations\n        for time_period in range(demand_array.shape[1]):  # Columns repres

In [16]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 7
All locations: 43
Warehouse/Supplier locations: 6 [0, 1, 2, 3, 4, 5]
Shelter locations: 21 [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
RSA locations: 9 [27, 28, 29, 30, 31, 32, 33, 34, 35]
PoD locations: 7 [36, 37, 38, 39, 40, 41, 42]
All resources: 6
Time periods: 12
Time Before Landfall: [0, 1, 2, 3, 4, 5]
Time After Landfall: [6, 7, 8, 9, 10, 11]
Scenario indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Scenario probabilities: [0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02]


In [17]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[23, 42]
Number of arcs: 1400
Number of arcs eliminated by this arc set: 449


In [18]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

# Initialize inventory list
Inv = []

# Define starting inventory based on instance version
if instance_version == "original":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]
    
elif instance_version == "largesize":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

elif instance_version == "smallsize":
    for i in range(V):
        Inv.append(df_V['Water StartInv'][i])
    I_ir0 = [[inv] for inv in Inv]

elif instance_version == "mediumsize":
    for i in range(V):
        Inv.extend([
            df_V['Water StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]


# Define the division factor
multiplier1 = 1  # Change this to the desired factor

# Read and divide the penalty costs
p_i = (df_V['P_i'] * multiplier1).tolist()  # Apply division and convert to a list


#time periods to travel i,j t_ijt[i][j]
seconds = 3600
t_ijt = [[math.ceil((timeMatrix[i][j] / seconds) / tperiod) for j in range(V)] for i in range(V)]

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

# Define the division factor
multilplier2 = 1  # Change this to the desired factor

#cost to travel i,j m_ij[i][j]
m_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        m_ij_value=((distanceMatrix[i][j]*fuel)+(t_ijt[i][j]*tp_salary*W_k)+(h_i[i]*tp_salary*W_k)) * multilplier2
        row.append(m_ij_value)
    m_ij.append(row)
    
#Cost to procure resource r from location i cost_ir[i][r]
cost=[]
for i in V0:
    cost.extend([0,0,df_V['Water PC'][i],df_V['Food PC'][i],0,0])
    cost_ir = [cost[i:i+R] for i in range(0, len(cost), R)]

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r = [1 / pallet for pallet in df_R['Pallet Units on Truck']]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter = df_R['pallet per S-Kit'].tolist()

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod = df_R['pallet per P-Kit'].tolist()

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i, j, t, capacity = row['From_Node'], row['To_Node'], row['Time_Period'], row['Capacity']
    K_ijt.setdefault(i, {}).setdefault(j, {})[t] = capacity

# Define the division factor
multiplier3 = 1  # Change this to any factor you want

# Iterate through K_ijt and divide each capacity by the factor
for i in K_ijt:
    for j in K_ijt[i]:
        for t in K_ijt[i][j]:
            K_ijt[i][j][t] *= multiplier3

#print("penalty cost=", p_i)
#print("m_ij=", m_ij)
#print(t_ijt)

In [19]:
start_time = time.time()  # Record the start time

# Define the model
m = gp.Model("Two-Stage Stochastic Programming Model")

# # Set the optimization direction
# m.modelSense = GRB.MINIMIZE

Set parameter Username
Set parameter LicenseID to value 2611420
Academic license - for non-commercial use only - expires 2026-01-17


In [20]:
# Define decision variables 
# i, j is position, r is commodity, t is time, and s is scenario

# Define first-stage decision variables
# I: Inventory of commodity r at node i at the start of time period t
I = {}

for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = m.addVar(vtype=GRB.CONTINUOUS, name="I_%s_%s_%s" % (i, r, t))

# Define second-stage decision variables
U = {}  # Demand shortage at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set (e.g., 0 and 1)
            U[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="U_%s_%s_%s" % (i, t, s))

S = {}  # Demand satisfied at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set
            S[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="S_%s_%s_%s" % (i, t, s))

f = {}  # Quantity of commodity r traveling on link (i, j) at time period t in scenario s

for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only define f_ijrt^s if (i, j) is a valid link
            for r in range(R):
                for t in range(T):
                    for s in sce:  # Iterate directly over the scenario set
                        f[i, j, r, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="f_%s_%s_%s_%s_%s" % (i, j, r, t, s))
m.update()

In [21]:
# Define constraints

# Add constraints for the lower bound of inventory at warehouses (V0)
for i in V0:  # Loop over warehouse nodes
    for r in range(R):  # Loop over commodities
        for t in range(1, T):  # Loop over time periods (from t=1 to T-1)
            # Calculate the total arc capacity for the current time period
            arc_capacity = sum(K_ijt[i][j][t] for j in range(V) if (i, j) in A)
            # Calculate max outflow for commodity r
            max_outflow_r = arc_capacity  / w_r[r]
            # Check if the initial inventory is greater than or equal to max_outflow_r
            if I_ir0[i][r] >= max_outflow_r:
                # Add the lower bound constraint
                m.addConstr(I[i, r, t] >= I_ir0[i][r] - max_outflow_r, name=f"LowerBound_Warehouse_{i}_{r}_{t}")


# Add constraints for the upper bound of inventory at PoDs (V3)
for i in V3:  # PoDs
    for r in range(R):  # Commodities
        for t in T_Bef[1:]:  # Time periods before landfall, excluding t=0
            # Calculate the total arc capacity for the current time period
            arc_capacity = sum(K_ijt[j][i][t] for j in range(V) if (j, i) in A)
            # Calculate max inflow for commodity r
            max_inflow_r = arc_capacity / w_r[r]            
            # Add the upper bound constraint
            m.addConstr(I[i, r, t] <= I_ir0[i][r] + max_inflow_r, name=f"UpperBound_PoD_{i}_{r}_{t}")


# Add constraints to ensure inventory at warehouses only decreases over time
for i in V0:
    for r in range(R):
        for t in range(1, T):
            m.addConstr(
                I[i, r, t] <= I[i, r, t-1],
                name=f"inventory_decrease_warehouse_{i}_{r}_{t}")

# Add constraints to ensure inventory at PoDs only increases before landfall
for i in V3:
    for r in range(R):
        for t in T_Bef[1:]:
            m.addConstr(
                I[i, r, t] >= I[i, r, t-1],
                name=f"inventory_increase_pod_{i}_{r}_{t}")

m.update()  # Ensure the model is updated with the latest constraints
#m.write("model_constraints.lp")

#Starting Value of I[t=0]
for i in range(V):
    for r in range(R):
        m.addConstr(I[i,r,0]==I_ir0[i][r])

# Update demand shortage for all locations, time periods (except t=0), and scenarios
for i in range(V):
    for t in range(1, T):
        for s in sce:
            m.addConstr(U[i, t, s] == U[i, t-1, s] - S[i, t, s] + d_its[(i, t, s)], 
                        name="DemandShortageUpdate_%s_%s_%s" % (i, t, s))

# Initial demand shortage for all locations and scenarios at time period t=0
for i in range(V):
    for s in sce:
        m.addConstr(U[i, 0, s] == d_its[(i, 0, s)], name="InitialDemandShortage_%s_%s" % (i, s))

# Inventory update constraint for all locations, commodities, time periods (except last), and scenarios
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                incoming_flow = gp.quicksum(f[j, i, r, t - t_ijt[j][i] + 1, s] for j in range(V) if (j, i) in A and t - t_ijt[j][i] + 1 >= 0)
                
                # Inventory update constraint for shelters
                if i in V1:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_shelter[r],
                        name="InventoryUpdate_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for pods
                elif i in V3:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_pod[r],
                        name="InventoryUpdate_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for warehouses and RSAs
                else:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow,
                        name="InventoryUpdate_%s_%s_%s_%s" % (i, r, t, s)
                    )


# Flow capacity constraint for all links, time periods, and scenarios
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in sce:
                    # Check if the capacity for this (i, j, t) exists in K_ijt
                    #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        m.addConstr(
                            gp.quicksum(w_r[r] * f[i, j, r, t, s] for r in range(R)) <= K_ijt[i][j][t],
                            name="FlowCapacity_%s_%s_%s_%s" % (i, j, t, s)
                        )

m.update()


In [22]:
# Set up the objective function

# Define the objective function
objective = gp.quicksum(
    P[s] * (
        gp.quicksum(
            gp.quicksum(p_i[i] * U[i, t, s] for i in range(V)) for t in range(T)
        ) +
        gp.quicksum(
            gp.quicksum(
                gp.quicksum(cost_ir[i][r] * f[i, j, r, t, s] for i in V0 for j in range(V) if (i, j) in A) +
                gp.quicksum(m_ij[i][j] * f[i, j, r, t, s] for i in range(V) for j in range(V) if (i, j) in A)
                for r in range(R)
            ) for t in range(T)
        )
    ) for s in sce
)

m.update()

m.setObjective(objective, GRB.MINIMIZE)
m.update()

In [23]:
# Optimize the model
m.optimize()

end_time = time.time()  # Record the end time

execution_time = end_time - start_time  # Calculate the execution time
print(f"Execution Time: {execution_time} seconds")

# Check if the optimization was successful
if m.status == GRB.OPTIMAL:

    # Initialize cost variables
    penalty_cost_before_landfall = 0
    penalty_cost_after_landfall = 0
    transportation_cost_before_landfall = 0
    transportation_cost_after_landfall = 0
    
    # Calculate costs for each time period subset
    for s in sce:
        for t in T_Bef:  # Before landfall
            for i in range(V):
                penalty_cost_before_landfall += P[s] * p_i[i] * U[i, t, s].X
    
            for r in range(R):
                for i in V0:
                    for j in range(V):
                        if (i, j) in A:
                            transportation_cost_before_landfall += P[s] * cost_ir[i][r] * f[i, j, r, t, s].X
    
                for i in range(V):
                    for j in range(V):
                        if (i, j) in A:
                            transportation_cost_before_landfall += P[s] * m_ij[i][j] * f[i, j, r, t, s].X
    
        for t in T_Aft:  # After landfall
            for i in range(V):
                penalty_cost_after_landfall += P[s] * p_i[i] * U[i, t, s].X
    
            for r in range(R):
                for i in V0:
                    for j in range(V):
                        if (i, j) in A:
                            transportation_cost_after_landfall += P[s] * cost_ir[i][r] * f[i, j, r, t, s].X
    
                for i in range(V):
                    for j in range(V):
                        if (i, j) in A:
                            transportation_cost_after_landfall += P[s] * m_ij[i][j] * f[i, j, r, t, s].X
    
    # Calculate total costs for each phase
    total_cost_before_landfall = penalty_cost_before_landfall + transportation_cost_before_landfall
    total_cost_after_landfall = penalty_cost_after_landfall + transportation_cost_after_landfall
    total_cost = total_cost_before_landfall + total_cost_after_landfall
    # Print the results
    print("Costs Before Landfall:")
    print(" - Penalty Cost:", penalty_cost_before_landfall)
    print(" - Transportation Cost:", transportation_cost_before_landfall)
    print(" - Total Cost:", total_cost_before_landfall)
    
    print("\nCosts After Landfall:")
    print(" - Penalty Cost:", penalty_cost_after_landfall)
    print(" - Transportation Cost:", transportation_cost_after_landfall)
    print(" - Total Cost:", total_cost_after_landfall)

    print("\nOverall Cost for both phases:", total_cost)

    """
    total_S_value = 0.0

    import numpy as np
    cost__S = np.zeros([len(sce), T])
    for t in range(T):
        for s in sce:
            cost__S[s, t] = sum([S[i, t, s].x for i in range(V)])
    # Output the total sum
    print("Total sum of S values:", cost__S)
    
    cost__U = np.zeros([len(sce), T])
    for t in range(T):
        for s in sce:
            cost__U[s, t] = sum([U[i, t, s].x for i in range(V)])
    # Output the total sum
    print("Total sum of U values:", cost__U)
    
    # Initialize cost variables
    total_penalty_cost = 0
    total_transportation_cost = 0

    # Calculate total penalty cost
    for s in sce:
        for t in range(T):
            for i in range(V):
                total_penalty_cost += P[s] * p_i[i] * U[i, t, s].X  # Use .X to get the optimized value of U

    # Calculate total transportation cost
    for s in sce:
        for t in range(T):
            for r in range(R):
                for i in V0:
                    for j in range(V):
                        if (i, j) in A:
                            total_transportation_cost += P[s] * cost_ir[i][r] * f[i, j, r, t, s].X

                for i in range(V):
                    for j in range(V):
                        if (i, j) in A:
                            total_transportation_cost += P[s] * m_ij[i][j] * f[i, j, r, t, s].X

    # Print the results
    print("Total Penalty Cost:", total_penalty_cost)
    print("Total Transportation Cost:", total_transportation_cost)
    print("Overall Total Cost:", total_penalty_cost + total_transportation_cost)
    
    
    # Create an empty list to store non-zero variable values
    variable_data = []

    # Loop through decision variables and get their values
    for var in m.getVars():
        if var.X != 0:  # Only save variables with non-zero values
            variable_data.append([var.VarName, var.X])  # Append variable name and value

    # Check if there are any non-zero variables to save
    if variable_data:
        # Create a DataFrame from the collected data
        df = pd.DataFrame(variable_data, columns=['Variable', 'Value'])

        # Save the DataFrame to an Excel file
        output_file = 'decision_variables_output.xlsx'
        df.to_excel(output_file, index=False, engine='openpyxl')

        print(f'Decision variable values have been saved to {output_file}')
    else:
        print("No non-zero variables found to save.")

    """
else:
    print("The model did not find an optimal solution.")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 10.0 (19045.2))

CPU model: 12th Gen Intel(R) Core(TM) i9-12900HK, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 1008774 rows, 5094696 columns and 14692480 nonzeros
Model fingerprint: 0xf035d38b
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [2e+00, 6e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e-01, 6e+05]
Presolve removed 99374 rows and 456468 columns
Presolve time: 6.42s
Presolved: 909400 rows, 4638228 columns, 14147790 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Elapsed ordering time = 5s
Elapsed ordering time = 22s
Elapsed ordering time = 25s
Elapsed ordering time = 30s
Elapsed ordering time = 35s
Elapsed ordering time = 40s
Ordering time: 43.91s
Elapsed ordering time = 46s
Elapsed ordering time = 50s
Elapsed ordering time 

In [24]:
#cost__S

In [25]:
#cost__U

In [26]:
#import pandas as pd
#pd.DataFrame(cost__S).plot.box()
#pd.DataFrame(cost__U).plot.box()